# 01. 데이터 전처리 및 스타일 라벨링

- `styles.csv`(Kaggle Fashion Product Images) 정제 → 규칙 기반 6개 스타일 라벨 부여 → Stratified 분할 (Train 70% / Val 10% / Test 20%)
- `fra_cleaned.csv`(Kaggle Fragrantica) 정제 → `accord_sequence`, 노트 가중 시퀀스(top×1, middle×2, base×3) → `fragrance_text` 생성
- Colab에서 실행 (파일 업로드 셀 포함)


In [ ]:
from google.colab import files
uploaded = files.upload()  # styles.csv, fra_cleaned.csv 업로드

In [ ]:
uploaded2 = files.upload()

In [9]:
"""
패션-향수 추천 프로젝트 전처리 및 학습 데이터 구축 파이프라인
========================================================================
1. styles.csv      - 패션 이미지 데이터셋 정제, 텍스트 정규화 및 스타일 라벨링
2. fra_cleaned.csv  - 향수 데이터셋 정제 및 조향학적 시계열 가중 감쇄 증강
3. Dataset Split    - 클래스 불균형 가중치 산출 및 Stratified Train/Val/Test 분할 (8:1:1)
"""

import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# ─────────────────────────────────────────────
# 텍스트 정규화
# ─────────────────────────────────────────────
def normalize_text(text: str) -> str:
    """소문자화 + 특수문자 제거 (쉼표·공백은 구분자로 보존)"""
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    # 알파벳, 숫자, 공백, 쉼표, 하이픈만 허용 (향 이름에 하이픈 많음)
    text = re.sub(r"[^a-z0-9\s,\-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ─────────────────────────────────────────────
# 룰 기반 스타일 분류 함수 (assign_style)
# ─────────────────────────────────────────────
def assign_style(row: pd.Series) -> str:
    usage   = str(row.get("usage", "")).lower().strip()
    article = str(row.get("articleType", "")).lower().strip()
    colour  = str(row.get("baseColour", "")).lower().strip()
    season  = str(row.get("season", "")).lower().strip()

    STREET_ARTICLES   = {'hoodies', 'sneakers', 'caps', 'jackets',
                         'backpacks', 'joggers', 'beanies', 'sweatshirts',
                         'track pants'}

    # Romantic: 색상 조건 제거, 아이템만으로 판정
    ROMANTIC_ARTICLES = {'dresses', 'skirts', 'heels', 'sarees', 'tops',
                         'flats', 'bra'}

    # Vintage: usage=ethnic 추가, 색상 조건 완화
    VINTAGE_ARTICLES  = {'coats', 'cardigans', 'vests', 'trench coats',
                         'loafers', 'brogues', 'waistcoat', 'kurtas',
                         'jeans', 'trousers', 'shirts'}
    VINTAGE_COLOURS   = {'brown', 'beige', 'khaki', 'mustard',
                         'olive', 'maroon', 'rust', 'dark green',
                         'navy blue', 'grey', 'charcoal'}

    if usage == 'formal':                                   return 'Formal'
    if usage == 'sports':                                   return 'Sporty'
    if usage == 'casual' and article in STREET_ARTICLES:   return 'Street'
    if usage in {'casual', 'smart casual'}:                return 'Casual'

    # Romantic: party이거나 아이템이 명확히 여성복인 경우
    if article in ROMANTIC_ARTICLES:                        return 'Romantic'

    # Vintage: ethnic usage 또는 색상+시즌 조합
    if usage == 'ethnic':                                   return 'Vintage'
    if (article in VINTAGE_ARTICLES
            and colour in VINTAGE_COLOURS):                 return 'Vintage'

    return None


# ─────────────────────────────────────────────
# 1. styles.csv 정제
# ─────────────────────────────────────────────
def clean_styles(input_path: str) -> pd.DataFrame:
    print("[1/3] styles.csv 정제 중...")

    # 쉼표 포함 셀로 인한 파싱 에러 방지: python engine + on_bad_lines='skip'
    df = pd.read_csv(
        input_path,
        engine="python",
        on_bad_lines="skip",   # 노이즈 행 무시
        dtype=str,             # 전체 str 로드 후 타입 후처리
    )
    original_len = len(df)
    print(f"  원본 행 수: {original_len:,}")

    # ── 필터링: 추천 도메인과 무관한 카테고리 제거 ──────────────────────
    # masterCategory: Apparel, Footwear 만 유지
    KEEP_MASTER = {"apparel", "footwear"}
    df["masterCategory_norm"] = df["masterCategory"].str.lower().str.strip()
    df = df[df["masterCategory_norm"].isin(KEEP_MASTER)].copy()
    print(f"  masterCategory 필터 후: {len(df):,}행 ({original_len - len(df):,}개 제거)")

    # subCategory: Socks, Wallets, Belts 등 스타일 판단에 불필요한 항목 제거
    DROP_SUB = {
        "socks", "wallets", "belts", "gloves", "keychains",
        "accessories", "shoe accessories", "shoe laces",
    }
    df["subCategory_norm"] = df["subCategory"].str.lower().str.strip()
    before = len(df)
    df = df[~df["subCategory_norm"].isin(DROP_SUB)].copy()
    print(f"  subCategory 필터 후: {len(df):,}행 ({before - len(df):,}개 제거)")

    # articleType: 소품류 / 문구류 / 기타 에지케이스 제거
    DROP_ARTICLE = {
        "pens", "key chain", "free gift", "ipad", "tablet sleeve",
        "cushion covers", "mens grooming kit", "body wash and scrub",
        "shoe laces", "hair accessory",
    }
    df["articleType_norm"] = df["articleType"].str.lower().str.strip()
    before = len(df)
    df = df[~df["articleType_norm"].isin(DROP_ARTICLE)].copy()
    print(f"  articleType 필터 후: {len(df):,}행 ({before - len(df):,}개 제거)")

    # ── 텍스트 정규화 ──────────────────────────────────────────────────
    TEXT_COLS = ["gender", "masterCategory", "subCategory", "articleType",
                 "baseColour", "season", "usage", "productDisplayName"]
    for col in TEXT_COLS:
        if col in df.columns:
            df[col] = df[col].apply(normalize_text)

    # ── id 정수 변환 ───────────────────────────────────────────────────
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    df = df.dropna(subset=["id"])
    df["id"] = df["id"].astype(int)

    # 임시 정규화 컬럼 제거
    df = df.drop(columns=["masterCategory_norm", "subCategory_norm", "articleType_norm"])

    print(f"  최종 styles 정제 행 수: {len(df):2,}\n")
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────
# 2. fra_cleaned.csv 정제 + 시계열 가중 감쇄 증강
# ─────────────────────────────────────────────
def parse_note_list(raw: str) -> list[str]:
    """'rose, jasmine, musk' → ['rose', 'jasmine', 'musk']"""
    if not isinstance(raw, str) or raw.strip() == "":
        return []
    items = re.split(r"[,;]+", raw.lower())
    cleaned = []
    for item in items:
        item = re.sub(r"[^a-z0-9\s\-]", "", item).strip()
        if item and item != "unknown":
            cleaned.append(item)
    return cleaned


def build_weighted_note_sequence(top: str, middle: str, base: str) -> str:
    """조향학적 시계열 가중치를 반영하여 base 노트를 3배 증강 매칭하는 시퀀스 생성"""
    top_notes    = parse_note_list(top)
    middle_notes = parse_note_list(middle)
    base_notes   = parse_note_list(base)

    tokens = []
    tokens.extend(top_notes)       # top: 1배
    tokens.extend(middle_notes * 2) # middle: 2배
    tokens.extend(base_notes * 3)   # base: 3배

    return " ".join(tokens)


def build_accord_sequence(row: pd.Series) -> str:
    """mainaccord1~5 에서 유효한 값만 추출하여 시퀀스 구성"""
    accords = []
    for i in range(1, 6):
        val = row.get(f"mainaccord{i}", "")
        if isinstance(val, str) and val.strip() and val.strip().lower() != "nan":
            acc = normalize_text(val)
            if acc:
                accords.append(acc)
    return ", ".join(accords)


def clean_fragrance(input_path: str) -> pd.DataFrame:
    print("[2/3] fra_cleaned.csv 정제 중...")

    df = pd.read_csv(
        input_path,
        sep=";",
        encoding="latin-1",
        engine="python",
        on_bad_lines="skip",
    )
    original_len = len(df)
    print(f"  원본 행 수: {original_len:,}")

    # ── 컬럼명 소문자 정규화 ──────────────────────────────────────────
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

    # ── 결측치 처리: NaN → 빈 문자열 (연산 에러 방지) ────────────────
    NOTE_COLS = ["top", "middle", "base",
                 "mainaccord1", "mainaccord2", "mainaccord3",
                 "mainaccord4", "mainaccord5"]
    for col in NOTE_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()

    # mainaccord1 이 완전히 비어있는 행은 향조 정보 없으므로 제거
    before = len(df)
    df = df[df["mainaccord1"].str.len() > 0].copy()
    print(f"  mainaccord 결측 제거 후: {len(df):,}행 ({before - len(df):,}개 제거)")

    # ── 텍스트 정규화 (note, accord 컬럼) ────────────────────────────
    for col in ["top", "middle", "base"]:
        df[col] = df[col].apply(normalize_text)
    for col in ["mainaccord1","mainaccord2","mainaccord3","mainaccord4","mainaccord5"]:
        df[col] = df[col].apply(normalize_text)

    # perfume, brand 소문자화
    for col in ["perfume", "brand", "country", "gender"]:
        if col in df.columns:
            df[col] = df[col].apply(normalize_text)

    # ── 조향학적 시계열 가중 감쇄 시퀀스 생성 ─────────────────────────
    df["weighted_note_sequence"] = df.apply(
        lambda r: build_weighted_note_sequence(r["top"], r["middle"], r["base"]),
        axis=1,
    )
    df["accord_sequence"] = df.apply(build_accord_sequence, axis=1)

    # 두 시퀀스를 결합한 최종 향수 텍스트 표현
    df["fragrance_text"] = (
        df["accord_sequence"] + " " + df["weighted_note_sequence"]
    ).str.strip()

    print(f"  최종 fragrance 정제 행 수: {len(df):,}\n")
    return df.reset_index(drop=True)


# ─────────────────────────────────────────────
# 3. 스타일 라벨 정의 및 데이터 셋 분할
# ─────────────────────────────────────────────
def build_style_labels(df: pd.DataFrame):
    print("[3/3] 스타일 라벨 재정의 및 학습셋 구축 중...")

    # assign_style 함수 기반 매핑 수행
    df['style_label'] = df.apply(assign_style, axis=1)

    # 미분류 제거
    before = len(df)
    df = df.dropna(subset=['style_label']).copy()
    print(f"  미분류 제거 후: {len(df):,}행 ({before - len(df):,}개 제거)")

    # 클래스 분포 확인
    print("\n  클래스 분포:")
    print(df['style_label'].value_counts())

    # 클래스 불균형 처리 (Loss function 전달용 가중치 산출)
    labels = df['style_label'].values
    classes = np.unique(labels)
    weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=labels
    )
    class_weight_dict = dict(zip(classes, weights))
    print(f"\n  클래스 가중치(Imbalance Weight): {class_weight_dict}")

    # Train/Val/Test 분할 (8:1:1) - 클래스 비율 유지를 위한 stratify 적용
    train_df, test_df = train_test_split(
        df, test_size=0.2,
        stratify=df['style_label'],
        random_state=42
    )
    train_df, val_df = train_test_split(
        train_df, test_size=0.125,
        stratify=train_df['style_label'],
        random_state=42
    )
    print(f"\n  분할 완료 -> Train: {len(train_df):,} / Val: {len(val_df):,} / Test: {len(test_df):,}")

    return train_df, val_df, test_df, class_weight_dict


# ─────────────────────────────────────────────
# 메인 실행 제어부
# ─────────────────────────────────────────────
STYLES_IN  = "styles.csv"
FRA_IN     = "fra_cleaned.csv"
OUT_DIR    = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# 1. 기존 데이터프레임 표준 전처리 연산
styles_clean = clean_styles(STYLES_IN)
fra_clean    = clean_fragrance(FRA_IN)

# 2. 스타일 라벨 재정의 및 8:1:1 계층적 데이터 세트 구축
train_df, val_df, test_df, class_weights = build_style_labels(styles_clean)

# 3. 정제 및 모델용 데이터 파티션 안전 저장
train_df.to_csv(os.path.join(OUT_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(OUT_DIR, "val.csv"), index=False)
test_df.to_csv(os.path.join(OUT_DIR, "test.csv"), index=False)
fra_clean.to_csv(os.path.join(OUT_DIR, "fra_preprocessed.csv"), index=False)

print("=" * 50)
print(f"🎉 모든 전처리 및 파티셔닝 파이프라인 컴파일 성공! 저장 경로: ./{OUT_DIR}")

[1/3] styles.csv 정제 중...
  원본 행 수: 44,424
  masterCategory 필터 후: 30,616행 (13,808개 제거)
  subCategory 필터 후: 30,604행 (12개 제거)
  articleType 필터 후: 30,604행 (0개 제거)
  최종 styles 정제 행 수: 30,604

[2/3] fra_cleaned.csv 정제 중...
  원본 행 수: 24,063
  mainaccord 결측 제거 후: 24,063행 (0개 제거)
  최종 fragrance 정제 행 수: 24,063

[3/3] 스타일 라벨 재정의 및 학습셋 구축 중...
  미분류 제거 후: 30,575행 (29개 제거)

  클래스 분포:
style_label
Casual      21572
Sporty       3626
Vintage      2476
Formal       1826
Street        546
Romantic      529
Name: count, dtype: int64

  클래스 가중치(Imbalance Weight): {'Casual': np.float64(0.23622442672600286), 'Formal': np.float64(2.7907082876962397), 'Romantic': np.float64(9.632955261499685), 'Sporty': np.float64(1.4053594410737267), 'Street': np.float64(9.333028083028083), 'Vintage': np.float64(2.0580910070005385)}

  분할 완료 -> Train: 21,402 / Val: 3,058 / Test: 6,115
🎉 모든 전처리 및 파티셔닝 파이프라인 컴파일 성공! 저장 경로: ./outputs


In [10]:
train_df, val_df, test_df, class_weights = build_style_labels(styles_clean)

[3/3] 스타일 라벨 재정의 및 학습셋 구축 중...
  미분류 제거 후: 30,575행 (29개 제거)

  클래스 분포:
style_label
Casual      21572
Sporty       3626
Vintage      2476
Formal       1826
Street        546
Romantic      529
Name: count, dtype: int64

  클래스 가중치(Imbalance Weight): {'Casual': np.float64(0.23622442672600286), 'Formal': np.float64(2.7907082876962397), 'Romantic': np.float64(9.632955261499685), 'Sporty': np.float64(1.4053594410737267), 'Street': np.float64(9.333028083028083), 'Vintage': np.float64(2.0580910070005385)}

  분할 완료 -> Train: 21,402 / Val: 3,058 / Test: 6,115
